# Notebook 05: MLOps - Pipeline de producción

## Objetivo

Documentar y demostrar como el modelo de lead scoring pasa de un notebook experimental 
a un sistema desplegable en producción. Cubrimos:

| Sección | Contenido |
|---------|----------|
| **5.1** | Pipeline end-to-end: de datos crudos a predicción |
| **5.2** | Tracking de experimentos con MLflow |
| **5.3** | Servir el modelo: API FastAPI + Docker |
| **5.4** | DAG 1 - Scoring pipeline (frecuencia alta) |
| **5.5** | DAG 2 - Monitoring + Retraining (frecuencia baja) |
| **5.6** | Arquitectura completa |
| **5.7** | Trabajo futuro y roadmap |

## Por qué MLOps?

Un modelo en un notebook no es un producto. Para que Raona pueda usar el lead scoring 
en su día a dia, necesita:
- Un **API** que reciba datos de un contacto y devuelva un score
- Un **contenedor** que se pueda desplegar en cualquier servidor
- **Orquestación** para procesar nuevos contactos automáticamente
- **Monitorización** para saber cuando el modelo deja de funciónar bien
- **Reproducibilidad** para poder reentrenar con datos nuevos

### Modelo en producción

En todo el pipeline de producción se usa exclusivamente el **modelo robusto** 
(`lead_scorer_robust.pkl`, 38 features). La única feature con data leakage 
(`nlp_contact_report_length`, derivada de `ai_CONTACT_REPORT` que se genera post-respuesta) 
queda excluida. El modelo completo (39 features) solo tiene valor academico y se documenta en NB04.

## Imports y configuración

In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import json
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

import lightgbm as lgb
import mlflow
import mlflow.sklearn

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = 'plotly_white'

# Rutas
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
WORKING_DATA = os.path.join(PROJECT_ROOT, '..', '_working', 'data')
WORKING_MODELS = os.path.join(PROJECT_ROOT, '..', '_working', 'models')
DELIVERABLE_MODELS = os.path.join(PROJECT_ROOT, 'app', 'models')
MLRUNS_DIR = os.path.join(PROJECT_ROOT, '..', '_working', 'mlruns')
APP_DIR = os.path.join(PROJECT_ROOT, 'app')

SEED = 42
np.random.seed(SEED)

print('Configuracion lista')
print(f'Modelos en: {DELIVERABLE_MODELS}')

/Users/acaballito/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Configuracion lista
Modelos en: /Users/acaballito/Library/CloudStorage/GoogleDrive-adriana.caballero@gmail.com/.shortcut-targets-by-id/1LzrxzfxIAZukyDLfOvioF7Z2ESoOXZGz/TFM/acaballero/TFM_deliverables/app/models


---
## 5.1 Pipeline end-to-end

Demostramos que el pipeline completo funcióna: desde un diccionario JSON con los datos 
de un contacto hasta la predicción del score, cluster y recomendación.

Esto es exactamente lo que hara el API en producción, usando únicamente el modelo robusto.

In [2]:
# Cargar artefactos del modelo robusto (produccion)
with open(os.path.join(DELIVERABLE_MODELS, 'preprocessor_robust.pkl'), 'rb') as f:
    preprocessor = pickle.load(f)

with open(os.path.join(DELIVERABLE_MODELS, 'lead_scorer_robust.pkl'), 'rb') as f:
    lead_scorer = pickle.load(f)

with open(os.path.join(DELIVERABLE_MODELS, 'clustering.pkl'), 'rb') as f:
    clustering_bundle = pickle.load(f)

with open(os.path.join(DELIVERABLE_MODELS, 'feature_names.pkl'), 'rb') as f:
    feature_names = pickle.load(f)

# Feature list del modelo robusto
if isinstance(feature_names, dict):
    FEATURE_COLS = feature_names['robust']
else:
    FEATURE_COLS = feature_names

print(f'Artefactos cargados:')
print(f'  Preprocessor: {type(preprocessor).__name__}')
print(f'  Lead scorer (robusto): {type(lead_scorer).__name__}')
print(f'  Features robustas: {len(FEATURE_COLS)}')
print(f'  Clustering: {list(clustering_bundle.keys())}')

Artefactos cargados:
  Preprocessor: Pipeline
  Lead scorer (robusto): LGBMClassifier
  Features robustas: 38
  Clustering: ['kmeans', 'imputer', 'scaler', 'features']


In [3]:
def score_lead(contact_data: dict) -> dict:
    """Pipeline completo: datos de contacto -> score + cluster + recomendacion.
    Usa exclusivamente el modelo robusto (sin features con data leakage).
    """
    # 1. Crear DataFrame
    df_input = pd.DataFrame([contact_data])
    for col in FEATURE_COLS:
        if col not in df_input.columns:
            df_input[col] = np.nan
    
    # 2. Preprocesar
    X = preprocessor.transform(df_input[FEATURE_COLS])
    
    # 3. Lead score
    score = lead_scorer.predict_proba(X)[:, 1][0]
    
    # 4. Cluster
    cluster_features = clustering_bundle['features']
    df_cluster = df_input[cluster_features].copy()
    X_cluster = clustering_bundle['scaler'].transform(
        clustering_bundle['imputer'].transform(df_cluster)
    )
    cluster = int(clustering_bundle['kmeans'].predict(X_cluster)[0])
    
    # 5. Clasificar riesgo
    if score >= 0.5:
        risk_level = 'ALTO'
    elif score >= 0.2:
        risk_level = 'MEDIO'
    else:
        risk_level = 'BAJO'
    
    return {
        'lead_score': round(float(score), 4),
        'cluster': cluster,
        'risk_level': risk_level,
        'recommended_channel': 'LinkedIn',  # Mayor reply rate: 5.0% vs 2.9%
        'recommended_day': 'Jueves',  # Mayor reply rate: 8.7%
    }

print('Funcion score_lead() definida')

Funcion score_lead() definida


In [4]:
# Demo: scorear un contacto real del pool de no contactados
pool_path = os.path.join(WORKING_DATA, 'not_contacted_pool_scored.parquet')
df_pool = pd.read_parquet(pool_path)

# Tomar el contacto con mayor score como ejemplo
top_contact = df_pool.nlargest(1, 'lead_score').iloc[0]
print(f'=== Demo: contacto real del pool (top 1 por lead_score) ===')
print(f'  Empresa: {top_contact.get("Company name", "N/A")}')
print(f'  Seniority: {top_contact.get("ai_SENIORITY", "N/A")}')
print(f'  Departamento: {top_contact.get("ai_DEPARTMENT", "N/A")}')
print()

# Scorear usando la funcion score_lead
demo_features = {f: top_contact[f] for f in FEATURE_COLS if f in top_contact.index}
result = score_lead(demo_features)
print('=== Resultado del scoring (modelo robusto) ===')
for k, v in result.items():
    print(f'  {k}: {v}')

=== Demo: contacto real del pool (top 1 por lead_score) ===
  Empresa: Real Madrid C.F.
  Seniority: MANAGER
  Departamento: Sales & MKT

=== Resultado del scoring (modelo robusto) ===
  lead_score: 0.887
  cluster: 1
  risk_level: ALTO
  recommended_channel: LinkedIn
  recommended_day: Jueves


In [5]:
# Verificar con datos reales del dataset
df = pd.read_parquet(os.path.join(WORKING_DATA, 'predictions.parquet'))

pos_sample = df[df['target_replied'] == 1].iloc[0]
neg_sample = df[df['target_replied'] == 0].iloc[0]

for label, sample in [('POSITIVO (respondio)', pos_sample), ('NEGATIVO (no respondio)', neg_sample)]:
    contact = {col: sample[col] for col in FEATURE_COLS if col in sample.index}
    result = score_lead(contact)
    print(f'\n=== Contacto {label} ===')
    print(f'  Score: {result["lead_score"]}')
    print(f'  Risk level: {result["risk_level"]}')
    print(f'  Cluster: {result["cluster"]}')


=== Contacto POSITIVO (respondio) ===
  Score: 0.7337
  Risk level: ALTO
  Cluster: 2

=== Contacto NEGATIVO (no respondio) ===
  Score: 0.1868
  Risk level: BAJO
  Cluster: 0


---
## 5.2 Tracking de experimentos con MLflow

MLflow permite registrar y comparar todos los experimentos realizados en NB04. 
Registramos retroactivamente los resultados de la competicion de modelos.

En producción, cada reentrenamiento del DAG 2 se registraria automáticamente en MLflow, 
creando un historial completo de versiones del modelo.

In [6]:
# Configurar MLflow con almacenamiento local
mlflow.set_tracking_uri(f'file://{os.path.abspath(MLRUNS_DIR)}')
experiment_name = 'raona_lead_scoring'
mlflow.set_experiment(experiment_name)

print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'Experimento: {experiment_name}')

MLflow tracking URI: file:///Users/acaballito/Library/CloudStorage/GoogleDrive-adriana.caballero@gmail.com/.shortcut-targets-by-id/1LzrxzfxIAZukyDLfOvioF7Z2ESoOXZGz/TFM/acaballero/_working/mlruns


Experimento: raona_lead_scoring


In [7]:
# Cargar metricas reales de NB04
import json as json_lib
metrics_path = os.path.join(WORKING_DATA, 'metrics.json')
with open(metrics_path) as f:
    nb04_metrics = json_lib.load(f)

complete_m = nb04_metrics['complete']
robust_m = nb04_metrics['robust']

# Registrar los experimentos de NB04 retroactivamente
experiments = [
    {
        'name': 'LightGBM_complete',
        'params': {'model_type': 'LightGBM', 'tuned': True, 'n_trials': 50,
                   'n_features': nb04_metrics['n_features_complete'],
                   'note': 'Modelo academico - incluye features con data leakage'},
        'metrics': {'pr_auc': complete_m['PR-AUC'], 'roc_auc': complete_m['ROC-AUC'],
                    'precision_at_100': complete_m['Precision@100'], 'lift_10pct': complete_m['Lift@10%']},
    },
    {
        'name': 'LightGBM_robust_PRODUCTION',
        'params': {'model_type': 'LightGBM', 'tuned': True,
                   'n_features': nb04_metrics['n_features_robust'],
                   'leakage_features_removed': ', '.join(nb04_metrics['leaking_features'])},
        'metrics': {'pr_auc': robust_m['PR-AUC'], 'roc_auc': robust_m['ROC-AUC'],
                    'precision_at_100': robust_m['Precision@100'], 'lift_10pct': robust_m['Lift@10%']},
    },
]

for exp in experiments:
    with mlflow.start_run(run_name=exp['name']):
        mlflow.log_params(exp['params'])
        mlflow.log_metrics(exp['metrics'])
        if exp['name'] == 'LightGBM_robust_PRODUCTION':
            mlflow.sklearn.log_model(lead_scorer, 'lead_scorer_robust')
    print(f'  Registrado: {exp["name"]} (PR-AUC={exp["metrics"]["pr_auc"]:.3f})')

print(f'\nTotal runs registrados: {len(experiments)}')

2026/03/17 12:21:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Registrado: LightGBM_complete (PR-AUC=0.303)


2026/03/17 12:21:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  Registrado: LightGBM_robust_PRODUCTION (PR-AUC=0.258)

Total runs registrados: 2


In [8]:
# Visualizar comparacion de experimentos
exp_df = pd.DataFrame([
    {'Modelo': e['name'], 'PR-AUC': e['metrics']['pr_auc'], 'ROC-AUC': e['metrics']['roc_auc']}
    for e in experiments
])

fig = go.Figure()
fig.add_trace(go.Bar(name='PR-AUC', x=exp_df['Modelo'], y=exp_df['PR-AUC'],
                     marker_color='#2ecc71', text=exp_df['PR-AUC'].round(3), textposition='auto'))
fig.add_trace(go.Bar(name='ROC-AUC', x=exp_df['Modelo'], y=exp_df['ROC-AUC'],
                     marker_color='#3498db', text=exp_df['ROC-AUC'].round(3), textposition='auto'))
fig.update_layout(title='MLflow: Comparacion de experimentos', barmode='group', height=400)
fig.show()

---
## 5.3 Servir el modelo: API FastAPI + Docker

Esta sección cubre el **Container 1** de la arquitectura: el servicio que esta siempre 
corriendo y atiende peticiones de scoring en tiempo real.

### Quien lo usa?
- El equipo comercial de Raona (via Streamlit o directamente)
- Integraciones con CRM u otros sistemás internos
- Cualquier sistema que necesite scorear un lead puntual

### Arquitectura

```
POST /score  ->  FastAPI  ->  modelo robusto  ->  JSON response
GET /health  ->  verificación de que el servicio esta activo
```

El contenedor Docker empaqueta el API + modelos + dependencias en una imagen 
reproducible que se despliega con `docker-compose up`.

> **Nota:** Las celdas siguientes generan los archivos en `app/` directamente desde el notebook
> para garantizar reproducibilidad. Los archivos generados son: `api.py`, `Dockerfile`,
> `docker-compose.yml` y `requirements.txt`.

In [9]:
# Generar api.py (usa exclusivamente el modelo robusto)
api_code = '''"""FastAPI endpoint para el lead scoring de Raona.

Uso:
    uvicorn api:app --host 0.0.0.0 --port 8000

Documentacion interactiva:
    http://localhost:8000/docs
"""
import os
import pickle
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional

# --- Cargar modelo robusto al iniciar ---
MODEL_DIR = os.path.join(os.path.dirname(__file__), "models")

with open(os.path.join(MODEL_DIR, "preprocessor_robust.pkl"), "rb") as f:
    preprocessor = pickle.load(f)
with open(os.path.join(MODEL_DIR, "lead_scorer_robust.pkl"), "rb") as f:
    lead_scorer = pickle.load(f)
with open(os.path.join(MODEL_DIR, "clustering.pkl"), "rb") as f:
    clustering_bundle = pickle.load(f)
with open(os.path.join(MODEL_DIR, "feature_names.pkl"), "rb") as f:
    feature_names = pickle.load(f)

FEATURE_COLS = feature_names["robust"] if isinstance(feature_names, dict) else feature_names
CLUSTER_FEATURES = clustering_bundle["features"]


# --- Schemas ---
class ContactInput(BaseModel):
    """Datos de entrada del contacto."""
    years_in_company: Optional[float] = Field(None, description="Anos en la empresa")
    number_of_connections: Optional[float] = Field(None, description="Conexiones en LinkedIn")
    number_of_employees: Optional[float] = Field(None, description="Empleados de la empresa")
    year_founded: Optional[float] = Field(None, description="Ano de fundacion")
    hiring_on_linkedin: Optional[float] = Field(None, description="1 si tiene ofertas activas")
    six_months_growth: Optional[float] = Field(None, description="Crecimiento plantilla 6 meses")
    two_years_growth: Optional[float] = Field(None, description="Crecimiento plantilla 2 anos")
    yearly_growth: Optional[float] = Field(None, description="Crecimiento plantilla anual")
    fe_seniority_ord: Optional[float] = Field(None, description="Seniority ordinal (0-5)")
    fe_type_of_contact_ord: Optional[float] = Field(None, description="Tipo contacto ordinal (0-5)")
    fe_fit_approved: Optional[float] = Field(None, description="FIT aprobado (0/1)")
    fe_fit_data_approved: Optional[float] = Field(None, description="FIT DATA aprobado (0/1)")
    fe_company_age: Optional[float] = Field(None, description="Edad de la empresa")
    fe_log_employees: Optional[float] = Field(None, description="log1p(empleados)")
    fe_company_size_bucket: Optional[float] = Field(None, description="Bucket tamano (0-4)")
    fe_log_connections: Optional[float] = Field(None, description="log1p(conexiones)")
    fe_headcount_momentum: Optional[float] = Field(None, description="Momentum crecimiento")
    fe_has_bio: Optional[float] = Field(None, description="Tiene bio en LinkedIn (0/1)")
    fe_microsoft_flag: Optional[float] = Field(None, description="Usa Microsoft (0/1)")
    fe_department_encoded: Optional[float] = Field(None, description="Departamento target-encoded")
    ext_ms_maturity_score: Optional[float] = Field(None, description="Score madurez Microsoft")
    ext_has_competitor_tech: Optional[float] = Field(None, description="Usa tech competidora (0/1)")
    nlp_report_length: Optional[float] = Field(None, description="Longitud company report")
    nlp_has_momentum: Optional[float] = Field(None, description="Tiene info momentum (0/1)")
    nlp_urgency_score: Optional[float] = Field(None, description="Score de urgencia")
    nlp_embedding_01: Optional[float] = Field(None, description="Embedding UMAP dim 1")
    nlp_embedding_02: Optional[float] = Field(None, description="Embedding UMAP dim 2")
    nlp_embedding_03: Optional[float] = Field(None, description="Embedding UMAP dim 3")
    nlp_topic: Optional[float] = Field(None, description="Topic cluster asignado")


class ScoreOutput(BaseModel):
    """Resultado del scoring."""
    lead_score: float = Field(description="Probabilidad de respuesta (0-1)")
    cluster: int = Field(description="Segmento asignado (0-6)")
    risk_level: str = Field(description="ALTO (>0.5), MEDIO (0.2-0.5), BAJO (<0.2)")
    recommended_channel: str = Field(description="Canal recomendado")
    recommended_day: str = Field(description="Mejor dia para contactar")


# --- App ---
app = FastAPI(
    title="Raona Lead Scoring API",
    description="Scoring de leads B2B con modelo robusto LightGBM",
    version="2.0.0",
)


@app.get("/health")
def health_check():
    return {
        "status": "ok",
        "model": "lead_scorer_robust",
        "n_features": len(FEATURE_COLS),
        "n_clusters": clustering_bundle["kmeans"].n_clusters,
    }


@app.post("/score", response_model=ScoreOutput)
def score_contact(contact: ContactInput):
    try:
        field_mapping = {
            "years_in_company": "Years in company",
            "number_of_connections": "Number of connections",
            "number_of_employees": "Number of employees",
            "year_founded": "Year founded",
            "hiring_on_linkedin": "Hiring on LinkedIn",
            "six_months_growth": "Six months headcount growth",
            "two_years_growth": "Two years headcount growth",
            "yearly_growth": "Yearly headcount growth",
        }
        contact_dict = {}
        for field_name, value in contact.dict().items():
            col_name = field_mapping.get(field_name, field_name)
            contact_dict[col_name] = value

        df_input = pd.DataFrame([contact_dict])
        for col in FEATURE_COLS:
            if col not in df_input.columns:
                df_input[col] = np.nan

        X = preprocessor.transform(df_input[FEATURE_COLS])
        score = float(lead_scorer.predict_proba(X)[:, 1][0])

        df_cluster = df_input[CLUSTER_FEATURES].copy()
        for col in CLUSTER_FEATURES:
            if col not in df_cluster.columns:
                df_cluster[col] = np.nan
        X_cluster = clustering_bundle["scaler"].transform(
            clustering_bundle["imputer"].transform(df_cluster)
        )
        cluster = int(clustering_bundle["kmeans"].predict(X_cluster)[0])

        risk_level = "ALTO" if score >= 0.5 else "MEDIO" if score >= 0.2 else "BAJO"

        return ScoreOutput(
            lead_score=round(score, 4), cluster=cluster, risk_level=risk_level,
            recommended_channel="LinkedIn", recommended_day="Jueves",
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''

api_path = os.path.join(APP_DIR, 'api.py')
with open(api_path, 'w') as f:
    f.write(api_code)

print(f'API generado: {api_path}')
print(f'Modelo: lead_scorer_robust.pkl ({len(FEATURE_COLS)} features, sin data leakage)')

API generado: /Users/acaballito/Library/CloudStorage/GoogleDrive-adriana.caballero@gmail.com/.shortcut-targets-by-id/1LzrxzfxIAZukyDLfOvioF7Z2ESoOXZGz/TFM/acaballero/TFM_deliverables/app/api.py
Modelo: lead_scorer_robust.pkl (38 features, sin data leakage)


#### Nota sobre el diseño de la API

La API actual es un **endpoint de scoring** que recibe features pre-procesadas 
(e.g., `fe_seniority_ord`, `nlp_embedding_01`). En un despliegue real, la cadena completa seria:

1. **Datos crudos** (perfil LinkedIn, tecnologias, informes AI) llegan via Airflow/ingestion
2. **Pipeline de feature engineering** (equivalente a NB03) transforma los datos crudos
3. **API de scoring** (este endpoint) recibe el vector de features y devuelve el score

Esta separación de responsabilidades es intencional: el feature engineering se ejecuta en batch 
(Airflow), mientras que el scoring es en tiempo real (API). La API no necesita conocer 
la logica de transformación, solo el formato de entrada del modelo.


In [10]:
# Simular una llamada POST /score
print('=== Simulacion de llamada POST /score ===')
print('\nRequest body (JSON):')
request_body = {
    'number_of_connections': 1200,
    'number_of_employees': 800,
    'fe_seniority_ord': 3,
    'fe_type_of_contact_ord': 4,
    'fe_company_size_bucket': 3,
    'fe_microsoft_flag': 1,
}
print(json.dumps(request_body, indent=2))

response = score_lead(request_body)
print('\nResponse:')
print(json.dumps(response, indent=2))

=== Simulacion de llamada POST /score ===

Request body (JSON):
{
  "number_of_connections": 1200,
  "number_of_employees": 800,
  "fe_seniority_ord": 3,
  "fe_type_of_contact_ord": 4,
  "fe_company_size_bucket": 3,
  "fe_microsoft_flag": 1
}

Response:
{
  "lead_score": 0.2975,
  "cluster": 0,
  "risk_level": "MEDIO",
  "recommended_channel": "LinkedIn",
  "recommended_day": "Jueves"
}


In [11]:
# Generar Dockerfile para Container 1 (API)
dockerfile = '''FROM python:3.9-slim

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \\
    libgomp1 \\
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY api.py .
COPY models/lead_scorer_robust.pkl models/
COPY models/preprocessor_robust.pkl models/
COPY models/clustering.pkl models/
COPY models/feature_names.pkl models/

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=5s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open(os.path.join(APP_DIR, 'Dockerfile'), 'w') as f:
    f.write(dockerfile)
print('Dockerfile generado (Container 1: API)')

Dockerfile generado (Container 1: API)


In [12]:
# Generar docker-compose.yml
compose = '''version: "3.8"

services:
  lead-scorer-api:
    build: .
    ports:
      - "8000:8000"
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 5s
      retries: 3
'''

with open(os.path.join(APP_DIR, 'docker-compose.yml'), 'w') as f:
    f.write(compose)

# Generar requirements.txt
requirements = '''fastapi==0.104.1
uvicorn==0.24.0
pandas==2.1.4
numpy==1.26.2
scikit-learn==1.5.2
lightgbm==4.5.0
pydantic==2.5.2
'''

with open(os.path.join(APP_DIR, 'requirements.txt'), 'w') as f:
    f.write(requirements)

print('docker-compose.yml generado')
print('requirements.txt generado')
print('\n=== Desplegar ===')
print('  cd app/')
print('  docker-compose up --build')
print('  # API en http://localhost:8000')
print('  # Docs en http://localhost:8000/docs')

docker-compose.yml generado
requirements.txt generado

=== Desplegar ===
  cd app/
  docker-compose up --build
  # API en http://localhost:8000
  # Docs en http://localhost:8000/docs


---
## 5.4 DAG 1: Scoring pipeline (frecuencia alta)

Este DAG se ejecuta **cada vez que Raona exporta contactos nuevos de Enginy** 
(tipicamente semanal). Su única función es procesar y scorear el batch de contactos.

### Flujo

```
ingest ──> transform ──> score ──> deliver
```

| Paso | Modulo | Que hace |
|------|--------|----------|
| **ingest** | `pipelines/ingest.py` | Carga CSV nuevo de Enginy, valida esquema, guarda Parquet |
| **transform** | `pipelines/transform.py` | Aplica feature engineering (mismas transformaciones que NB03) |
| **score** | `pipelines/score.py` | Scorea todos los contactos con `lead_scorer_robust.pkl`, asigna clusters |
| **deliver** | `pipelines/score.py` | Genera CSV rankeado por score para el equipo comercial |

### Resultado

Un archivo CSV (o tabla en base de datos) con todos los contactos nuevos ordenados 
por `lead_score` descendente, con su cluster, nivel de riesgo y recomendaciones.

### Frecuencia

Semanal o bajo demanda. Se puede disparar manualmente o programar en Airflow.

---
## 5.5 DAG 2: Monitoring + Retraining (frecuencia baja)

Este DAG se ejecuta **mensualmente**. Su función es vigilar si el modelo sigue siendo 
fiable y, si detecta cambios significativos en los datos, reentrenar automáticamente.

### Por qué es necesario?

El modelo se entreno con datos de un período concreto. Con el tiempo:
- Raona puede empezar a vender productos nuevos que atraen otro perfil
- El mercado cambia y responden perfiles diferentes
- LinkedIn modifica su algoritmo, alterando tasas de respuesta

El modelo sigue scoreando, pero sus predicciones pueden dejar de ser fiables. 
Este DAG lo detecta antes de que sea un problema.

### Flujo

```
evaluate_drift ──> check_drift
                       │
               ┌──────┴──────┐
               │             │
         PSI > 0.25    PSI <= 0.25
               │             │
           retrain       skip (fin)
               │
           validate
               │
            deploy
```

| Paso | Modulo | Que hace |
|------|--------|----------|
| **evaluate_drift** | `pipelines/monitor.py` | Calcula PSI por feature comparando datos recientes vs entrenamiento |
| **check_drift** | DAG logic | Si PSI > 0.25 en 3+ features: reentrenar. Si no: fin |
| **retrain** | `pipelines/retrain.py` | Reentrena LightGBM con datos acumulados (solo modelo robusto) |
| **validate** | `pipelines/validate.py` | Compara nuevo modelo vs actual en holdout. Solo promueve si es mejor |
| **deploy** | `pipelines/validate.py` | Reemplaza `lead_scorer_robust.pkl` en el Container 1 |

### Frecuencia

Mensual. Podrian pasar meses sin reentrenar si los datos son estables.

### Implementación del PSI (Population Stability Index)

El PSI compara la distribución de cada feature entre los datos de entrenamiento 
y los datos más recientes:

| PSI | Interpretación | Acción |
|-----|---------------|--------|
| < 0.10 | Sin cambio significativo | Nada |
| 0.10 - 0.25 | Cambio moderado | Monitorizar |
| > 0.25 | Cambio significativo | Considerar reentrenamiento |

In [13]:
def calculate_psi(expected, actual, bins=10):
    """Calcula el Population Stability Index entre dos distribuciones."""
    breakpoints = np.percentile(expected[~np.isnan(expected)],
                                np.linspace(0, 100, bins + 1))
    breakpoints = np.unique(breakpoints)
    
    if len(breakpoints) < 3:
        return 0.0
    
    expected_counts = np.histogram(expected[~np.isnan(expected)], bins=breakpoints)[0]
    actual_counts = np.histogram(actual[~np.isnan(actual)], bins=breakpoints)[0]
    
    expected_pct = expected_counts / max(expected_counts.sum(), 1) + 1e-6
    actual_pct = actual_counts / max(actual_counts.sum(), 1) + 1e-6
    
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

print('Funcion calculate_psi() definida')

Funcion calculate_psi() definida


In [14]:
# Simular monitorizacion: dividir datos en referencia y produccion
df = pd.read_parquet(os.path.join(WORKING_DATA, 'predictions.parquet'))

n_half = len(df) // 2
df_train_ref = df.iloc[:n_half]
df_prod_sim = df.iloc[n_half:]

# Nota: como no disponemos de una columna temporal, dividimos el dataset por posicion.
# Esto equivale a un split aleatorio, no temporal. En produccion, el PSI se calcularia
# entre los datos de entrenamiento originales y los datos nuevos del ultimo mes.
print(f'Datos de referencia (entrenamiento): {len(df_train_ref):,} filas')
print(f'Datos simulados (produccion): {len(df_prod_sim):,} filas')

# Calcular PSI para cada feature del modelo robusto
psi_results = []
for col in FEATURE_COLS:
    if col in df.columns:
        psi = calculate_psi(df_train_ref[col].values, df_prod_sim[col].values)
        status = 'OK' if psi < 0.1 else ('MONITORIZAR' if psi < 0.25 else 'ALERTA')
        psi_results.append({'Feature': col, 'PSI': psi, 'Status': status})

psi_df = pd.DataFrame(psi_results).sort_values('PSI', ascending=False)
print('\n=== PSI por feature (modelo robusto) ===')
print(psi_df.to_string(index=False))

Datos de referencia (entrenamiento): 2,993 filas
Datos simulados (produccion): 2,994 filas



=== PSI por feature (modelo robusto) ===
                    Feature      PSI      Status
           fe_log_employees 0.181015 MONITORIZAR
        Number of employees 0.181015 MONITORIZAR
           nlp_embedding_01 0.158765 MONITORIZAR
           nlp_embedding_02 0.149021 MONITORIZAR
     fe_company_size_bucket 0.146012 MONITORIZAR
      fe_headcount_momentum 0.123971 MONITORIZAR
     fe_type_of_contact_ord 0.116110 MONITORIZAR
 Two years headcount growth 0.109045 MONITORIZAR
Six months headcount growth 0.105107 MONITORIZAR
           nlp_embedding_03 0.099063          OK
      fe_department_encoded 0.077968          OK
    Yearly headcount growth 0.077795          OK
      ext_ms_maturity_score 0.072872          OK
               Year founded 0.062031          OK
             fe_company_age 0.053566          OK
                  nlp_topic 0.035623          OK
          nlp_urgency_score 0.029411          OK
          nlp_report_length 0.018129          OK
      Number of connections

In [15]:
# Visualizar PSI
colors = ['#e74c3c' if s == 'ALERTA' else '#f39c12' if s == 'MONITORIZAR' else '#2ecc71'
          for s in psi_df['Status']]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=psi_df['Feature'], y=psi_df['PSI'],
    marker_color=colors,
    text=psi_df['PSI'].round(3), textposition='auto'
))
fig.add_hline(y=0.1, line_dash='dash', line_color='#f39c12',
              annotation_text='Umbral monitorizar (0.1)')
fig.add_hline(y=0.25, line_dash='dash', line_color='#e74c3c',
              annotation_text='Umbral alerta (0.25)')
fig.update_layout(
    title='Population Stability Index (PSI) por feature',
    xaxis_title='Feature', yaxis_title='PSI',
    height=500, xaxis_tickangle=-45
)
fig.show()

In [16]:
# Simular drift artificial para demostrar la deteccion
print('=== Simulacion de drift artificial ===')
print('Escenario: Number of employees aumenta un 50% en datos nuevos\n')

df_drift = df_prod_sim.copy()
df_drift['Number of employees'] = df_drift['Number of employees'] * 1.5

psi_normal = calculate_psi(
    df_train_ref['Number of employees'].values,
    df_prod_sim['Number of employees'].values
)
psi_drift = calculate_psi(
    df_train_ref['Number of employees'].values,
    df_drift['Number of employees'].values
)

print(f'PSI sin drift:  {psi_normal:.4f} -> {"OK" if psi_normal < 0.1 else "ALERTA"}')
print(f'PSI con drift:  {psi_drift:.4f} -> {"OK" if psi_drift < 0.1 else "ALERTA"}')
print(f'\nEl DAG 2 detectaria este cambio y dispararia el reentrenamiento.')

=== Simulacion de drift artificial ===
Escenario: Number of employees aumenta un 50% en datos nuevos

PSI sin drift:  0.1810 -> ALERTA
PSI con drift:  0.3891 -> ALERTA

El DAG 2 detectaria este cambio y dispararia el reentrenamiento.


---
## 5.6 Arquitectura completa

### Como encajan las piezas

```
Enginy (exporta CSV nuevos)
         │
    DAG 1: Scoring (semanal)
         │
         ├── ingest > transform > score > deliver
         │                                  │
         │                         CSV rankeado para
         │                         equipo comercial
         │
    DAG 2: Monitoring (mensual)
         │
         ├── evaluate_drift > check
         │                     │
         │              si drift: retrain > validate > deploy
         │                                              │
         │                                    actualiza .pkl
         │                                              │
    Container 1: API (siempre activo)  <────────────────┘
         │
         ├── POST /score (scoring individual)
         ├── Streamlit conecta aquí
         └── CRM / integraciones
```

### Estructura de archivos

```
TFM_deliverables/
├── app/
│   ├── api.py                        # FastAPI (Container 1)
│   ├── streamlit_app.py              # App interactiva (3 tabs)
│   ├── Dockerfile                    # Container 1: API
│   ├── docker-compose.yml
│   ├── requirements.txt
│   ├── models/
│   │   ├── lead_scorer_robust.pkl    # Modelo de PRODUCCION
│   │   ├── lead_scorer_complete.pkl  # Solo uso academico
│   │   ├── preprocessor_robust.pkl
│   │   ├── preprocessor.pkl
│   │   ├── clustering.pkl
│   │   └── feature_names.pkl
│   ├── dags/
│   │   ├── scoring_dag.py            # DAG 1: Scoring semanal
│   │   └── monitoring_dag.py         # DAG 2: Monitoring mensual
│   └── pipelines/
│       ├── ingest.py
│       ├── transform.py
│       ├── score.py
│       ├── monitor.py
│       ├── retrain.py
│       └── validate.py
├── notebooks/
│   ├── 01_data_loading.ipynb
│   ├── 02_eda.ipynb
│   ├── 03_feature_engineering.ipynb
│   ├── 04_models.ipynb
│   └── 05_mlops.ipynb
├── report/
│   ├── index.html
│   ├── nb01-nb05 HTML exports
│   └── charts/
└── _working/
    ├── data/
    └── mlruns/
```

---
## 5.7 Trabajo futuro y roadmap de producción

### Mejoras del modelo

| Mejora | Justificacion | Esfuerzo |
|--------|--------------|----------|
| Datos externos INE (sector/digitalizacion) | Enriquecer features ext_ con datos publicos del sector para mejorar el perfilado de empresas | Medio |
| Datos de engagement previo | Si Raona tiene historico de interacciones previas con una empresa (reuniones, propuestas), incorporarlo como feature de relacion previa mejoraria la prediccion | Medio |

### Integración con Enginy

1. **Exportacion programatica**: script que descarga nuevos contactos via API de Enginy
2. **Scoring automático**: DAG 1 se dispara al detectar nuevo CSV
3. **Priorizacion en campaña**: ordenar contactos por lead_score antes del outreach
4. **Feedback loop**: resultados del outreach retroalimentan el reentrenamiento

### A/B Testing

Para validar el impacto real del modelo robusto:
- **Grupo A**: Outreach priorizado por lead_score (top 50%)
- **Grupo B**: Outreach aleatorio (baseline actual de Raona)
- **Métrica**: reply rate del grupo A vs grupo B
- **Duracion**: 4-6 semanas, 500 contactos por grupo
- **Hipotesis**: basandonos en el lift@10% de 2.1x, esperamos que el grupo A tenga una tasa de respuesta significativamente superior

---
## Resumen

### Artefactos generados

| Archivo | Descripción |
|---------|------------|
| `app/api.py` | Endpoint FastAPI (POST /score, GET /health) |
| `models/lead_scorer_robust.pkl` | **Modelo de PRODUCCION** (38 features, sin data leakage) |
| `models/preprocessor_robust.pkl` | Pipeline preprocesado del modelo robusto |
| `app/Dockerfile` | Container 1: imagen Docker para la API |
| `app/docker-compose.yml` | Orquestación del contenedor |
| `app/requirements.txt` | Dependencias |
| `_working/mlruns/` | Registro de experimentos MLflow |

### Arquitectura de producción

| Componente | Función | Frecuencia |
|-----------|---------|------------|
| **Container 1 (API)** | Scoring individual via REST | Siempre activo |
| **DAG 1 (Scoring)** | Batch scoring de contactos nuevos | Semanal |
| **DAG 2 (Monitoring)** | Vigilancia de drift + reentrenamiento | Mensual |
| **Streamlit** | Interfaz visual para el equipo comercial | Siempre activo |

### Modelo robusto en producción

| Métrica | Valor |
|---------|-------|
| PR-AUC | 0.258 |
| Precisión@100 | 31% (vs 14% aleatorio) |
| Lift@10% | 2.1x |
| Features | 38 (1 excluida por data leakage) |

### Conclusión

El sistema va más alla de un modelo aislado: es un **pipeline de producción profesional** 
con scoring en tiempo real (API), procesamiento en batch (DAG 1), monitorización continua 
(DAG 2) y una interfaz accesible (Streamlit). El modelo robusto duplica la tasa de respuesta 
en el top 10% de leads, y Raona podria desplegarlo con `docker-compose up`.

In [17]:
# Inventario final
print('=' * 60)
print('RESUMEN NOTEBOOK 05 - MLOps')
print('=' * 60)

generated_files = [
    ('app/api.py', 'FastAPI endpoint (modelo robusto)'),
    ('app/Dockerfile', 'Container 1: imagen Docker'),
    ('app/docker-compose.yml', 'Orquestacion'),
    ('app/requirements.txt', 'Dependencias'),
    ('app/models/lead_scorer_robust.pkl', 'Modelo robusto (PRODUCCION)'),
    ('app/models/preprocessor_robust.pkl', 'Pipeline preprocesado robusto'),
    ('app/models/clustering.pkl', 'K-Means + scaler'),
    ('app/models/feature_names.pkl', 'Lista de features'),
]

print('\nArchivos del sistema de produccion:')
for path, desc in generated_files:
    full_path = os.path.join(PROJECT_ROOT, path)
    exists = 'OK' if os.path.exists(full_path) else 'PENDIENTE'
    print(f'  [{exists}] {path} -- {desc}')

print('\nMLflow experiments:')
print(f'  URI: {mlflow.get_tracking_uri()}')
print(f'  Runs registrados: {len(experiments)}')

print(f'\nModelo en produccion: lead_scorer_robust.pkl ({len(FEATURE_COLS)} features)')
print('Feature excluida (data leakage): nlp_contact_report_length (de ai_CONTACT_REPORT, post-respuesta)')

RESUMEN NOTEBOOK 05 - MLOps

Archivos del sistema de produccion:
  [OK] app/api.py -- FastAPI endpoint (modelo robusto)
  [OK] app/Dockerfile -- Container 1: imagen Docker


  [OK] app/docker-compose.yml -- Orquestacion
  [OK] app/requirements.txt -- Dependencias
  [OK] app/models/lead_scorer_robust.pkl -- Modelo robusto (PRODUCCION)
  [OK] app/models/preprocessor_robust.pkl -- Pipeline preprocesado robusto
  [OK] app/models/clustering.pkl -- K-Means + scaler
  [OK] app/models/feature_names.pkl -- Lista de features

MLflow experiments:
  URI: file:///Users/acaballito/Library/CloudStorage/GoogleDrive-adriana.caballero@gmail.com/.shortcut-targets-by-id/1LzrxzfxIAZukyDLfOvioF7Z2ESoOXZGz/TFM/acaballero/_working/mlruns
  Runs registrados: 2

Modelo en produccion: lead_scorer_robust.pkl (38 features)
Feature excluida (data leakage): nlp_contact_report_length (de ai_CONTACT_REPORT, post-respuesta)
